# 01 - Data Exploration

Explore the transaction-level data, identify inconsistencies and fix clear data errors.

Ground rules for this notebook:
- We only **fix clear, rule-based errors** here (duplicates, impossible values). Anything that depends on the data distribution (imputation, outlier capping, scaling) is only *identified* here and will be done inside the sklearn pipeline later, to avoid leakage.
- `spend_next_90d` is never used as a feature for classification.
- Features must be computable at purchase time, using only that row, the training data and `clientes.csv`.

## 1. Loading the data and understanding the columns

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = "../data"

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
clientes = pd.read_csv(f"{DATA_DIR}/clientes.csv")

print("train:   ", train.shape)
print("test:    ", test.shape)
print("clientes:", clientes.shape)

train:    (6534, 40)
test:     (986, 38)
clientes: (1143, 13)


In [3]:
# Which columns are in train but not in test (should be exactly the two targets)
print("Only in train:", sorted(set(train.columns) - set(test.columns)))
print("Only in test: ", sorted(set(test.columns) - set(train.columns)))

Only in train: ['repeat_purchase_90d', 'spend_next_90d']
Only in test:  []


In [4]:
train.head()

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,spend_next_90d,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise
0,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,1,359.04,79.0,126.0,2.0,858.30,429.150000,112.656252,349.49,508.81,47.000000,NaN,0.0,0.00,1.0,349.49,2.0,858.30,2.0,858.30,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,10000000,-0.660448
1,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,1,592.71,244.0,616.0,5.0,6722.83,1344.566000,1683.795178,84.38,4222.15,93.000000,117.657696,0.0,0.00,0.0,0.00,0.0,0.00,4.0,2500.68,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,10000001,0.924510
2,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,1,423.72,9.0,610.0,90.0,17590.08,195.445333,190.389932,4.54,1002.15,6.752809,6.384073,2.0,111.27,9.0,1928.55,28.0,5290.25,49.0,7899.80,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,10000002,-0.258730
3,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,0,NaN,97.0,97.0,1.0,270.33,270.330000,NaN,270.33,270.33,NaN,NaN,0.0,0.00,0.0,0.00,1.0,270.33,1.0,270.33,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,10000003,1.448402
4,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,1,455.73,85.0,1575.0,25.0,12803.02,512.120800,406.696963,22.58,1736.74,62.083333,66.246520,0.0,0.00,2.0,184.13,4.0,2062.50,4.0,2062.50,122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,10000004,-1.058890


### Matching columns to the variable table in the brief

The brief describes the columns in groups. Several names are in Portuguese, so we translate them here:

| Brief group | Columns | Meaning |
|---|---|---|
| Identifiers | `ID`, `customer_id` | Row / submission ID; customer key to join `clientes.csv` |
| Targets | `repeat_purchase_90d`, `spend_next_90d` | Classification target (0/1); regression target in EUR (train only) |
| Date | `purchase_date` | Anonymised date; order and intervals preserved |
| Current monetary values | `Total a Pagar` (total to pay), `Total Bruto` (gross), `Total Liquido` (net), `Total Imp_` (tax), `Total Portes` (shipping), `Total Desc_ Global` (global discount), `Total Desc_ Linha` (line discounts), `Custo` (cost) | Values of the current transaction |
| Current purchase details | `N_ Detalhes` (number of detail lines), `invoice_count_current_purchase` | Size of the current purchase |
| Current categoricals | `payment_type`, `salesperson`, `commercial_zone`, `transport_method`, `warehouse`, `document_series` | |
| Historical timing | `days_since_previous_purchase`, `customer_tenure_days`, `prior_purchase_count`, `prior_mean_gap_days`, `prior_std_gap_days` | Recency, tenure, frequency, gaps between past purchases |
| Historical value | `prior_total_spend`, `prior_mean/std/min/max_purchase_value` | Summary of past purchase values |
| Rolling frequency | `purchase_count_30d/90d/180d/365d` | Number of purchases in the previous N days |
| Rolling spend | `spend_30d/90d/180d/365d` | Spend in the previous N days |
| Noise | `random_noise` | To be checked for usefulness |

We store these groups in a dictionary so later sections can refer to them by name.

In [5]:
column_groups = {
    "identifiers": ["ID", "customer_id"],
    "targets": ["repeat_purchase_90d", "spend_next_90d"],
    "date": ["purchase_date"],
    "current_monetary": ["Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_",
                         "Total Portes", "Total Desc_ Global", "Total Desc_ Linha", "Custo"],
    "current_details": ["N_ Detalhes", "invoice_count_current_purchase"],
    "categoricals": ["payment_type", "salesperson", "commercial_zone",
                     "transport_method", "warehouse", "document_series"],
    "hist_timing": ["days_since_previous_purchase", "customer_tenure_days", "prior_purchase_count",
                    "prior_mean_gap_days", "prior_std_gap_days"],
    "hist_value": ["prior_total_spend", "prior_mean_purchase_value", "prior_std_purchase_value",
                   "prior_min_purchase_value", "prior_max_purchase_value"],
    "rolling_count": ["purchase_count_30d", "purchase_count_90d", "purchase_count_180d", "purchase_count_365d"],
    "rolling_spend": ["spend_30d", "spend_90d", "spend_180d", "spend_365d"],
    "noise": ["random_noise"],
}

# Sanity check: every train column belongs to exactly one group
grouped = [c for cols in column_groups.values() for c in cols]
print("Columns in train but not in any group:", set(train.columns) - set(grouped))
print("Columns in a group but not in train:  ", set(grouped) - set(train.columns))
print("Columns listed twice:                 ", len(grouped) - len(set(grouped)))

Columns in train but not in any group: set()
Columns in a group but not in train:   set()
Columns listed twice:                  0


### Data types and missing values per column

One table with the dtype in train and test plus the % of missing values. If a column has a different dtype in train and test, it is usually a sign of missing values (pandas stores an integer column with NaNs as float) or of unexpected text in a numeric column.

In [8]:
overview = pd.DataFrame({
    "dtype_train": train.dtypes,
    "dtype_test": test.dtypes.reindex(train.columns),
    "missing_train": train.isna().sum(),
    "missing_test": test.isna().sum().reindex(train.columns),
    "missing_%_train": (train.isna().mean() * 100).round(1),
    "missing_%_test": (test.isna().mean().reindex(train.columns) * 100).round(1),
    "n_unique_train": train.nunique(),
})
overview

,dtype_train,dtype_test,missing_train,missing_test,missing_%_train,missing_%_test,n_unique_train
Total a Pagar,float64,float64,65,0.0,1.0,0.0,6122
Total Bruto,float64,float64,65,0.0,1.0,0.0,6038
Total Liquido,float64,float64,65,0.0,1.0,0.0,6093
Total Imp_,float64,float64,65,0.0,1.0,0.0,5406
Total Portes,float64,float64,65,0.0,1.0,0.0,215
Total Desc_ Global,float64,float64,65,0.0,1.0,0.0,109
Total Desc_ Linha,float64,float64,65,0.0,1.0,0.0,3519
Custo,float64,float64,65,0.0,1.0,0.0,5736
N_ Detalhes,float64,int64,65,0.0,1.0,0.0,171
invoice_count_current_purchase,float64,int64,65,0.0,1.0,0.0,5


In [9]:
# Columns whose dtype differs between train and test
overview[overview["dtype_train"] != overview["dtype_test"]].dropna(subset=["dtype_test"])

,dtype_train,dtype_test,missing_train,missing_test,missing_%_train,missing_%_test,n_unique_train
N_ Detalhes,float64,int64,65,0.0,1.0,0.0,171
invoice_count_current_purchase,float64,int64,65,0.0,1.0,0.0,5
customer_tenure_days,float64,int64,65,0.0,1.0,0.0,2121
prior_purchase_count,float64,int64,65,0.0,1.0,0.0,277
purchase_count_30d,float64,int64,65,0.0,1.0,0.0,12
purchase_count_90d,float64,int64,65,0.0,1.0,0.0,23
purchase_count_180d,float64,int64,65,0.0,1.0,0.0,38
purchase_count_365d,float64,int64,65,0.0,1.0,0.0,61


### clientes.csv

Optional customer-level table, one row per customer (to be confirmed in Section 7).

In [10]:
clientes.head()

,customer_id,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address,customer_type,company_name,customer_since_year
0,842570,Country CT11: Land of The Final Version,Region RG37: We Will See Later,NaN,NaN,City CY417: Missing Cell Port,NaN,Postal code PC828: #REF! Delivery Route,NaN,Address AD1120: Cell Without a Home,NaN,Excel Is Not A Database Ltd 150,2018
1,770941,Country CT9: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,Postal code PC56: #REF! Delivery Route,NaN,Address AD62: Cell Without a Home,NaN,Excel Is Not A Database Ltd 1064,2021
2,827053,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC827: #REF! Delivery Route,Street ST170: PowerPoint New Walk,Address AD449: Cell Without a Home,NaN,Excel Is Not A Database Ltd 113,2016
3,494648,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD454: Cell Without a Home,NaN,Excel Is Not A Database Ltd 430,2015
4,429291,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD453: Cell Without a Home,NaN,Excel Is Not A Database Ltd 673,2015


In [ ]:
pd.DataFrame({
    "dtype": clientes.dtypes,
    "missing": clientes.isna().sum(),
    "missing_%": (clientes.isna().mean() * 100).round(1),
    "n_unique": clientes.nunique(),
})

,dtype,missing,missing_%,n_unique
customer_id,int64,0,0.0,1143
country_or_region,str,0,0.0,11
commercial_region,str,114,10.0,37
island,str,578,50.6,10
municipality,str,159,13.9,183
city,str,546,47.8,417
postal_area,str,71,6.2,271
postal_code,str,170,14.9,828
street,str,598,52.3,511
address,str,6,0.5,1120


: 

### Key insight - Section 1

*(to be written after running the cells above)*